# 03 · Gold: dimensions
Builds the dimension tables of the sales star schema from silver. Dimensions use the source business keys (`ProductKey`, `ResellerKey`, ...) as their keys, so facts join to them directly.

| Table | Grain |
| --- | --- |
| `dim_product` | one row per product |
| `dim_reseller` | one row per reseller |
| `dim_sales_territory` | one row per sales territory |
| `dim_salesperson` | one row per salesperson |
| `dim_date` | one row per calendar day covering all order and target dates |
| `bridge_salesperson_territory` | salesperson-to-territory assignments |

In [0]:
%run ../../config/pipeline_config

In [0]:
%run ../../utils/common_functions

In [0]:
def silver(name):
    return spark.table(f"{SILVER_SCHEMA}.{name}")

dim_product = silver("product").select(
    "ProductKey", "ProductName", "Category", "Subcategory", "Color", "StandardCost")

dim_reseller = silver("reseller").select(
    "ResellerKey", "ResellerName", "BusinessType", "City", "StateProvince", "CountryRegion")

dim_sales_territory = silver("region").select(
    "SalesTerritoryKey", "Region", "Country", F.col("Group").alias("TerritoryGroup"))

dim_salesperson = silver("salesperson").select(
    "EmployeeKey", "EmployeeID", "SalespersonName", "Title", "UPN")

bridge_salesperson_territory = silver("salesperson_region").select("EmployeeKey", "SalesTerritoryKey")

In [0]:
bounds = (silver("sales").select(F.col("OrderDate").alias("d"))
          .unionByName(silver("targets").select(F.col("TargetMonth").alias("d")))
          .agg(F.trunc(F.min("d"), "year").alias("start"),
               F.last_day(F.add_months(F.trunc(F.max("d"), "year"), 11)).alias("end")))

dim_date = (bounds
    .select(F.explode(F.sequence("start", "end", F.expr("interval 1 day"))).alias("Date"))
    .select(
        F.date_format("Date", "yyyyMMdd").cast("int").alias("DateKey"),
        "Date",
        F.year("Date").alias("Year"),
        F.quarter("Date").alias("Quarter"),
        F.month("Date").alias("MonthNumber"),
        F.date_format("Date", "MMMM").alias("MonthName"),
        F.date_format("Date", "yyyy-MM").alias("YearMonth"),
        F.trunc("Date", "month").alias("MonthStartDate"),
        F.weekofyear("Date").alias("WeekOfYear"),
        F.dayofmonth("Date").alias("DayOfMonth"),
        F.date_format("Date", "EEEE").alias("DayName"),
        F.dayofweek("Date").isin(1, 7).alias("IsWeekend"),   # 1 = Sunday, 7 = Saturday
    ))

In [0]:
dimensions = {
    "dim_product":                  (dim_product, ["ProductKey"]),
    "dim_reseller":                 (dim_reseller, ["ResellerKey"]),
    "dim_sales_territory":          (dim_sales_territory, ["SalesTerritoryKey"]),
    "dim_salesperson":              (dim_salesperson, ["EmployeeKey"]),
    "dim_date":                     (dim_date, ["DateKey"]),
    "bridge_salesperson_territory": (bridge_salesperson_territory, ["EmployeeKey", "SalesTerritoryKey"]),
}

dq = DataQualityChecker("gold")
for name, (df, keys) in dimensions.items():
    dq.not_null(name, df, keys)
    dq.unique(name, df, keys)
dq.assert_no_critical_failures()   # never publish a dimension with duplicate keys

for name, (df, _) in dimensions.items():
    write_delta_table(df.withColumn("gold_load_timestamp", F.current_timestamp()),
                      f"{GOLD_SCHEMA}.{name}", f"{GOLD_PATH}{name}")